# Rotation Correlation Curve Analysis

Standalone debug viewer for the **2D SOFT rotation correlation** (all angles in **radians**). It reads the CSV dumps that the fs2d debug run already produces (no C++ changes needed).

## Workflow
1. Run a registration with the debug flag on (service or test executable, `useDirect=true` for the coefficient files).
2. In this notebook click **Run All**.
3. Zoom into any region of the curve (rangeslider at the bottom, or the lo/hi sliders) to inspect its curvature.
4. The last section fits the curve with the **true kernel** and runs a **hidden-component scan** that can reveal *shoulder/plateau* features which classical peak detection cannot see (e.g. the ~0.55 rad / GT region of the earlier pair 285->290).

## Files read from `./data`
| file | content |
|---|---|
| `rotationCorrelation1D.csv` | correlation curve `[index, angle(rad), normalizedCorrelation]` |
| `rotationPeaks.csv` | detected peaks `[angle, peakCorrelation, covariance, levelPotential, index]` |
| `registration_meta.csv` | pair + estimated rotation + GT error (one row) |
| `dataForReadIn.csv` | parameters (N, thresholds, numAngles, numTotalSolutions) |
| `sigCoefR/I_1angle.csv`, `patCoefR/I_1angle.csv` | spherical-harmonic coefficients (written by the `useDirect=true` debug path) -> true kernel |

Requirements: `numpy`, `plotly`, `ipywidgets` (same env as `plot_2d_registration.ipynb`).


In [173]:
"""Imports + data location."""
import os
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown

# Data directory: this notebook lives in plotting_results/2d, debug output in ./data
DATA_DIR = os.path.join(os.getcwd(), "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print("DATA_DIR:", DATA_DIR)


DATA_DIR: /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data


In [174]:
"""Robust loaders (tolerate headers, tab/comma/space delimiters, trailing non-numeric tokens)."""

def load_lines(fname):
    """All parsed rows (list of lists of floats); headers / non-numeric rows skipped."""
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        return None
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            vals = []
            for t in ln.replace(",", " ").split():
                try:
                    vals.append(float(t))
                except ValueError:
                    break
            if vals:
                out.append(vals)
    return out

def load_csv(fname):
    """2D array using the dominant column count (e.g. registration_meta keeps its 13 numeric cols)."""
    rows = load_lines(fname)
    if not rows:
        return None
    counts = [len(r) for r in rows]
    modal = max(set(counts), key=counts.count)
    rows = [r for r in rows if len(r) == modal]
    return np.array(rows) if rows else None

def status(name, arr):
    print("  [%s] %s%s" % ("ok" if arr is not None else "MISSING", name,
                           "  (%d rows)" % arr.shape[0] if arr is not None and arr.ndim == 2 else ""))


In [175]:
"""Load everything and print a summary (angles in rad)."""
print("Loading debug files from", DATA_DIR)
curve = load_csv("rotationCorrelation1D.csv")   # [index, angle(rad), normalizedCorrelation]
peaks = load_csv("rotationPeaks.csv")           # [angle, peakCorrelation, covariance, levelPotential, index]
meta  = load_csv("registration_meta.csv")       # frame1 frame2 rot_angle_deg tx ty ...
cfg   = load_lines("dataForReadIn.csv")         # parameter rows of mixed width

status("rotationCorrelation1D.csv", curve)
status("rotationPeaks.csv", peaks)
status("registration_meta.csv", meta)
import time as _t
for _f in ("rotationCorrelation1D.csv", "rotationPeaks.csv", "registration_meta.csv"):
    _p = os.path.join(DATA_DIR, _f)
    if os.path.isfile(_p):
        print("    mtime %s: %s" % (_f, _t.strftime("%Y-%m-%d %H:%M:%S", _t.localtime(os.path.getmtime(_p)))))

# --- parameters (dataForReadIn: 6-col metadata row + per-angle 3-col rows) ---
N = num_angles = num_total = None
level_thresh = None
if cfg:
    for r in cfg:
        if len(r) == 6:
            N = int(r[0]); _cn = int(r[1]); _cell = r[2]
            level_thresh = r[3]; num_angles = int(r[4]); num_total = int(r[5])
            break
print("\nParameters: N=%s  level_potential=%s  numAngles=%s  numTotalSolutions=%s" % (N, level_thresh, num_angles, num_total))

# --- ground truth (derived: GT = estimated rotation + GT error), converted to rad ---
gt_rad, pair = None, None
est_rad = gt_err_rad = gt_minus_rad = None
if meta is not None and len(meta) >= 1:
    pair = (int(meta[0, 0]), int(meta[0, 1]))
    est_deg = float(meta[0, 2])
    gt_err_deg = float(meta[0, 7]) if meta.shape[1] > 7 else 0.0
    est_rad = np.deg2rad(est_deg)
    gt_err_rad = np.deg2rad(gt_err_deg)
    gt_rad = est_rad + gt_err_rad
    gt_minus_rad = est_rad - gt_err_rad          # alternative sign convention: error = est - true
    print("Pair: %d -> %d   estimated rotation: %.4f rad   GT error: %.4f rad   -> GT(est+err): %.4f rad"
          % (pair[0], pair[1], est_rad, gt_err_rad, gt_rad))
else:
    print("No registration_meta.csv -> ground truth marker will not be shown.")

# --- detected peaks ---
if peaks is not None:
    print("\nDetected rotation peaks (from rotationPeaks.csv):")
    print("  %10s %11s %13s" % ("angle(rad)", "correlation", "levelPotential"))
    for p in peaks:
        print("  %10.4f %11.4f %13.4f" % (p[0], p[1], p[3]))
    print("  Note: every peak has an antipodal copy at +pi rad (the curve is pi-periodic).")
    print("  Physical rotations are angles modulo pi rad.")


Loading debug files from /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data
  [ok] rotationCorrelation1D.csv  (4096 rows)
  [ok] rotationPeaks.csv  (8 rows)
  [ok] registration_meta.csv  (1 rows)
    mtime rotationCorrelation1D.csv: 2026-08-27 16:58:14
    mtime rotationPeaks.csv: 2026-08-27 16:58:14
    mtime registration_meta.csv: 2026-08-27 16:58:18

Parameters: N=256  level_potential=0.01  numAngles=8  numTotalSolutions=321
Pair: 2070 -> 2075   estimated rotation: 0.0966 rad   GT error: 0.1611 rad   -> GT(est+err): 0.2578 rad

Detected rotation peaks (from rotationPeaks.csv):
  angle(rad) correlation levelPotential
      1.1290      0.3150        0.0992
      1.6690      0.8336        0.6949
      3.2382      0.7565        0.5723
      3.5849      1.0000        1.0000
      4.2706      0.3150        0.0992
      4.8106      0.8336        0.6949
      0.0966      0.7566        0.5725
      0.4433      1.0000        0.9999
  Note: every peak has an antipodal copy a

In [176]:
"""Build the (interactive) correlation curve figure."""
import plotly.graph_objects as go

fig1 = go.Figure()   # plain Figure: no anywidget dependency
if curve is None:
    print("No correlation curve available.")
else:
    x, c = curve[:, 1], curve[:, 2]

    # robust fold: uniform [0, 2pi) grid with an even number of samples
    n = len(x)
    step = np.median(np.diff(x))
    uniform = np.allclose(np.diff(x), step, atol=1e-9)
    if not uniform or not np.isclose(x[0], 0.0, atol=1e-6):
        xg = np.linspace(0.0, 2 * np.pi, n)
        c = np.interp(xg, x, c)
        x = xg
    n = len(x) - (len(x) % 2)
    x, c = x[:n], c[:n]
    half = n // 2
    fold_x = x[:half]
    fold_c = (c[:half] + c[half:]) / 2.0

    fig1.add_trace(go.Scatter(x=x, y=c, name="correlation C(theta)",
                              line=dict(color="steelblue", width=2),
                              hovertemplate="%{x:.4f} rad<br>corr %{y:.4f}<extra></extra>"))
    fig1.add_trace(go.Scatter(x=fold_x, y=fold_c, name="folded [0,pi) = (C+C(pi))/2",
                              line=dict(color="orange", width=1.5, dash="dot"), visible="legendonly"))

    if peaks is not None:
        fig1.add_trace(go.Scatter(x=peaks[:, 0], y=peaks[:, 1], mode="markers+text",
                                  name="detected peaks",
                                  marker=dict(symbol="x", size=12, color="red", line=dict(width=2)),
                                  text=["%.4f rad" % a for a in peaks[:, 0]],
                                  textposition="top center", textfont=dict(size=10, color="red")))

    if gt_rad is not None:
        fig1.add_vline(x=gt_rad, line=dict(color="green", dash="dash", width=1.5),
                       annotation_text="GT(est+err) %.4f" % gt_rad, annotation_position="top left")
    if gt_minus_rad is not None:
        fig1.add_vline(x=float(np.mod(gt_minus_rad, 2 * np.pi)), line=dict(color="orange", dash="dash", width=1.5),
                       annotation_text="GT(est-err) %.4f" % gt_minus_rad, annotation_position="top left")

    cm = c.min() if len(c) else 0.0
    fig1.update_layout(
        title="Rotation correlation curve, pair %d -> %d" % (pair[0], pair[1]) if pair else "Rotation correlation curve",
        height=520,
        legend=dict(orientation="h", y=1.12, font=dict(size=11)),
        xaxis=dict(title="rotation angle (rad)", rangeslider=dict(visible=True),
                   range=[0, 2 * np.pi], showgrid=True,
                   tickvals=[0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi],
                   ticktext=["0", "pi/2", "pi", "3pi/2", "2pi"]),
        yaxis=dict(title="normalized correlation", range=[min(cm, 0) - 0.05, 1.05]),
        margin=dict(t=80))
    print("Tip: drag the rangeslider at the bottom to zoom, or use the sliders below.")


Tip: drag the rangeslider at the bottom to zoom, or use the sliders below.


In [177]:
"""Zoom helpers: set the visible x-range of the figure above (re-rendered via an Output widget)."""
lo = widgets.FloatSlider(value=0.0, min=0.0, max=2 * np.pi, step=0.01, description="lo (rad)")
hi = widgets.FloatSlider(value=2 * np.pi, min=0.0, max=2 * np.pi, step=0.01, description="hi (rad)")
btn = widgets.Button(description="Apply zoom")
reset = widgets.Button(description="Reset")
out = widgets.Output()

def _redraw():
    with out:
        out.clear_output(wait=True)
        display(fig1)

def _apply(_b):
    a, b = sorted([lo.value, hi.value])
    fig1.update_xaxes(range=[a, b])
    _redraw()

def _reset(_b):
    fig1.update_xaxes(range=[0, 2 * np.pi])
    _redraw()

btn.on_click(_apply)
reset.on_click(_reset)
_redraw()
display(widgets.VBox([widgets.HBox([lo, hi]), widgets.HBox([btn, reset]), out]))


## Kernel fit & hidden-component scan

**What is the "true kernel"?** A matched rotation peak does not look Gaussian in this pipeline: its shape is the autocorrelation of the resampled reference descriptor (a band-limited bump with tails), computable exactly and for free from the SH coefficients of the debug dump:

$$K(\theta) = \sum_{m \neq 0} \left(\sum_{\ell} |\hat b_{\ell m}|^2\right) \cos(m\theta)$$

**Why fold?** The curve is (nearly) exactly pi-periodic: $C(\theta+\pi)=C(\theta)$. Everything lives in $[0, \pi)$; antipodal copies need no special handling.

**The scan (GT-free):** classical persistence detection only supplies the *initial hypotheses*. The kernel fit then probes a window around **every** persistence peak ($\pm$ `WIN_HALF_RAD`) for ONE additional component - a *shoulder/plateau* that is not a local maximum and is invisible to classical peak detection (e.g. the ~0.55 rad plateau of pair 285->290, which was the true rotation yet invisible). A candidate counts only if it survives the non-negative fit AND improves the residual by >= `MIN_IMPROV_RATIO`; the strongest accepted candidate becomes the hidden plateau. GT is only read from the meta row for a final "did we pick correctly" line - it never steers the scan.

Tune `WIN_HALF_RAD`, `WIN_CENTER_RAD` (set a number to force a single window), `MIN_SEP_RAD`, `KNOWN_MARGIN_RAD` in the next cell if needed.


In [178]:
"""True kernel from the SH coefficients of the reference (scan 2) descriptor + scan parameters (in rad)."""
# --- fit parameters (all in radians) --------------------------------
COARSE_RAD = 0.01        # hidden-component scan grid step (~0.57 deg)
FINE_RAD = 0.0004        # local refinement step (~0.02 deg)
MIN_SEP_RAD = 0.1        # min separation between components (~5.7 deg)
WIN_CENTER_RAD = None    # None = GT-free per-peak scan (window around EVERY persistence peak)
                         # set a number (e.g. GT mod pi) to force a single window (debug)
WIN_HALF_RAD = 0.35      # scan window half width (~20 deg)
MIN_IMPROV_RATIO = 2.0   # min residual improvement for a hidden candidate to be accepted
MAX_HIDDEN = 2        # max additional components tested per window (each level must justify itself)
KNOWN_MARGIN_RAD = 0.26  # persistence peaks within +/-this of the window are known components

c2R = load_csv("patCoefR_1angle.csv")
c2I = load_csv("patCoefI_1angle.csv")
c1R = load_csv("sigCoefR_1angle.csv")
c1I = load_csv("sigCoefI_1angle.csv")

def build_kernel_acf(cR, cI, theta):
    """Autocorrelation kernel of a descriptor from its SH coefficients:
    K(theta) = sum_{m!=0} (sum_l |c_lm|^2) cos(m theta).
    The coefficient array uses the alm layout of softRegistrationClass.cpp
    (see the almIdx formula there); the file is a raw dump of that array."""
    bw = int(round(np.sqrt(len(cR))))
    bigL = bw - 1
    Q = np.zeros(2 * bw)
    for l in range(bw):
        for m in range(-l, l + 1):
            if m >= 0:
                idx = m * (bigL + 1) - m * (m - 1) // 2 + (l - m)
            else:
                idx = bigL * (bigL + 3) // 2 + 1 + (bigL + m) * (bigL + m + 1) // 2 + (l - abs(m))
            Q[m + bw] += cR[idx] ** 2 + cI[idx] ** 2
    mpos = np.arange(1, bw)                       # Q is symmetric in m (descriptor is real)
    K = 2.0 * (Q[mpos + bw] @ np.cos(np.outer(mpos, theta)))
    return K

def build_kernel_empirical(theta):
    """Fallback: measure the peak shape from the strongest peak of the observed
    folded curve itself (mirrored, 0.9 rad wide)."""
    Fk = fold_c
    i0 = int(np.argmax(Fk))
    W = int(round(0.9 / (theta[1] - theta[0])))
    seg = Fk[max(0, i0 - W):i0 + W + 1]
    K = np.zeros(len(theta))
    for j in range(W + 1):
        v = (seg[W - j] + (seg[W + j] if W + j < len(seg) else 0.0)) / 2.0
        K[j] = v
        if j > 0:
            K[len(theta) - j - 1] = v
    return K

if curve is None:
    print("Kernel panel skipped: no correlation curve.")
    K = None
elif c2R is not None and c2I is not None:
    print("Building true kernel K from scan-2 descriptor (patCoef, %d coefficients)" % len(c2R))
    K = build_kernel_acf(c2R[:, 0], c2I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
elif c1R is not None and c1I is not None:
    print("WARNING: patCoef missing -> using scan-1 descriptor (sigCoef) as kernel.")
    K = build_kernel_acf(c1R[:, 0], c1I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
else:
    print("WARNING: no coefficient files -> measuring the kernel empirically from the dominant peak.")
    K = build_kernel_empirical(x)
    K = (K - K.min()) / (K.max() - K.min())


Building true kernel K from scan-2 descriptor (patCoef, 16384 coefficients)


In [179]:
"""GT-free hidden-component scan: for EVERY persistence peak (folded mod pi),
scan its window (peak +- WIN_HALF_RAD) for ONE additional kernel component.

Classical persistence detection supplies the initial rotation hypotheses; the
kernel fit then tests each neighborhood. A candidate counts only if it keeps a
positive amplitude AND improves the residual by >= MIN_IMPROV_RATIO. The
strongest accepted candidate becomes the hidden plateau. GT (meta row) is only
used for a final validation line, never to center a window."""
mus_known, amps_known = [], []
hidden_result = None
kernel_peaks = []          # per fitted component: mu, amp, hidden?, shoulder?, anchor
scan_rad, scan_resid = [], []
peak_scan = []             # per persistence peak: window scan result dict

def circ_dist(a, b, period=np.pi):
    d = np.abs(np.asarray(a) - b) % period
    return np.minimum(d, period - d)

def kval(a):
    w = np.mod(np.asarray(a), 2 * np.pi)
    return np.interp(np.minimum(w, 2 * np.pi - w), x, K)   # K is even on [0, 2pi)

def design_matrix(angles, th):
    return np.column_stack([kval(th - a) for a in angles] + [np.ones(len(th))])

def fit_nonneg(th, Fw, angles_in):
    """Least squares with non-negative component amplitudes, affine baseline.
    Returns (coef, r, kept_angles); negative-amplitude components are dropped."""
    kept = list(angles_in)
    while True:
        A = design_matrix(kept, th)
        coef, res, *_ = np.linalg.lstsq(A, Fw, rcond=None)
        r = res[0] if len(res) else float(np.sum((A @ coef - Fw) ** 2))
        amps = coef[:-1]
        if len(amps) == 0 or np.all(amps >= 0):
            return coef, r, kept
        del kept[int(np.argmin(amps))]

def scan_window(ctr, th, F, known_all):
    """Scan ONE window centered at ctr (rad, folded circle) for up to
    MAX_HIDDEN additional components. Level k is accepted only if its MARGINAL
    residual improvement (resid_{k-1} / resid_k) >= MIN_IMPROV_RATIO, so every
    added peak must justify itself. Returns a result dict with n_hidden (0/1/2)
    or None when the window is empty / holds no valid candidate."""
    lo, hi = ctr - WIN_HALF_RAD, ctr + WIN_HALF_RAD          # may cross the 0/pi seam
    mask = circ_dist(th, ctr) < WIN_HALF_RAD                 # circular window mask on [0, pi)
    thw, Fw = th[mask], F[mask]
    if len(thw) < 100:
        return None
    win_samp = th[mask]                                      # window samples (both seam pieces)
    m0 = [m for m in known_all if np.min(circ_dist(m, win_samp)) < KNOWN_MARGIN_RAD]
    m0 = list(dict.fromkeys(m0))
    coef_k, r_k, kept_k = fit_nonneg(thw, Fw, m0)
    knowns = list(kept_k)
    seg_lo, seg_hi = max(lo, 0.0), min(hi, np.pi)            # main (interior) piece
    cand = np.arange(seg_lo + COARSE_RAD, seg_hi - COARSE_RAD, COARSE_RAD)
    if hi > np.pi:                                           # wrapped piece near 0
        cand = np.concatenate([np.arange(0.0, hi - np.pi - COARSE_RAD, COARSE_RAD), cand])
    elif lo < 0.0:                                           # wrapped piece near pi
        cand = np.concatenate([np.arange(np.pi + lo + COARSE_RAD, np.pi, COARSE_RAD), cand])
    if len(cand) == 0:
        return None
    wrapped = (lo < 0.0) or (hi > np.pi)

    def grid_resids(angles, excl):
        res = []
        for mu in cand:
            if any(circ_dist(mu, m) < MIN_SEP_RAD for m in excl):
                res.append(np.inf)
                continue
            coef, r, kept = fit_nonneg(thw, Fw, angles + [mu])
            res.append(r if kept == angles + [mu] else np.inf)
        return np.array(res)

    def refine_all(hiddens):
        """Alternating local refinement of every hidden position (each stays
        inside the window and >= MIN_SEP_RAD away from knowns and the others)."""
        cur = list(hiddens)
        coef, r_cur, kept = fit_nonneg(thw, Fw, knowns + cur)
        while True:
            changed = False
            for idx in range(len(cur)):
                others = knowns + [cur[j] for j in range(len(cur)) if j != idx]
                c0 = cur[idx]
                for a in np.arange(max(c0 - 0.044, cand[0]), min(c0 + 0.044, cand[-1]), FINE_RAD):
                    if any(circ_dist(a, o) < MIN_SEP_RAD for o in others):
                        continue
                    trial = list(cur); trial[idx] = float(a)
                    coef2, r2, kept2 = fit_nonneg(thw, Fw, knowns + trial)
                    if kept2 != knowns + trial or r2 >= r_cur:
                        continue
                    cur[idx] = float(a); r_cur = r2; changed = True
            if not changed:
                break
        coef, r, kept = fit_nonneg(thw, Fw, knowns + cur)
        return cur, r, coef, kept

    hiddens, coef, kept = [], coef_k, knowns
    r_fin = r_k
    r1a = r2a = None
    mu1 = mu2 = None
    n_hidden = 0
    res1 = grid_resids(knowns, knowns)
    if not np.isinf(res1).all():
        mu1 = cand[int(np.argmin(res1))]
        hiddens, r_fin, coef, kept = refine_all([mu1])
        r1a = r_fin
        if r_k / r_fin >= MIN_IMPROV_RATIO:
            n_hidden = 1
            if MAX_HIDDEN >= 2:
                res2 = grid_resids(knowns + hiddens, knowns + hiddens)
                if not np.isinf(res2).all():
                    mu2 = cand[int(np.argmin(res2))]
                    hid2, r2, coef2, kept2 = refine_all(hiddens + [mu2])
                    if r_fin / r2 >= MIN_IMPROV_RATIO:
                        hiddens, r_fin, coef, kept = hid2, r2, coef2, kept2
                        r2a = r_fin
                        n_hidden = 2
    lm = [th[i] for i in range(1, len(th) - 1) if F[i] > F[i - 1] and F[i] > F[i + 1]]
    mus_all = knowns + hiddens
    sh = [not any(circ_dist(m, L) < 0.026 for L in lm) for m in hiddens]
    return dict(center=float(ctr),
                lo=float(max(lo, 0.0)), hi=float(min(hi, np.pi)),
                wrapped=bool(wrapped),
                n_hidden=n_hidden,
                mu=float(hiddens[0]) if n_hidden else (float(mu1) if mu1 is not None else lo),
                amp=float(coef[len(knowns)]) if len(hiddens) >= 1 else 0.0,
                mu2=float(hiddens[1]) if n_hidden > 1 else None,
                amp2=float(coef[len(knowns) + 1]) if n_hidden > 1 else 0.0,
                ratio1=(r_k / r1a) if r1a is not None else None,
                ratio2=(r1a / r2a) if r2a is not None else None,
                resid0=r_k, resid1=float(r1a) if r1a is not None else r_k, resid2=float(r2a) if r2a is not None else None,
                ratio=(r_k / r_fin) if r_fin < r_k else 1.0,
                shoulder=bool(sh[0]) if sh else None,
                shoulder2=bool(sh[1]) if len(sh) > 1 else None,
                coef=coef, mus_all=mus_all, knowns=knowns,
                cand=cand, resids=res1)
if K is not None:
    th, F = fold_x, fold_c                         # folded grid [0, pi)
    known_all = []
    if peaks is not None:
        for p in peaks[:, 0]:
            m = float(p) % np.pi
            if not any(circ_dist(m, u) < 2e-4 for u in known_all):
                known_all.append(m)
        known_all.sort()
    print("Persistence peaks on [0, pi): %s" % [round(m, 4) for m in known_all])

    if WIN_CENTER_RAD is not None:
        centers = [float(WIN_CENTER_RAD % np.pi)]
        print("Scan mode: SINGLE window at %.4f rad (debug override WIN_CENTER_RAD)" % centers[0])
    else:
        centers = list(known_all)                   # GT-free: one window per persistence peak
        print("Scan mode: GT-FREE per-peak scan over %d window(s)" % len(centers))

    for ctr in centers:
        r = scan_window(ctr, th, F, known_all)
        if r is None:
            print("  window at %.4f rad: empty / no valid candidate" % ctr)
            continue
        r["accepted"] = r["n_hidden"] > 0
        peak_scan.append(r)
        if r["accepted"]:
            extra = "   + 2nd hidden at %.4f rad (marginal %.1fx)" % (r["mu2"], r["ratio2"]) if r["n_hidden"] > 1 else ""
            wrap_txt = " (wraps the 0/pi seam)" if r.get("wrapped") else ""
            print("  anchor %.4f rad, window [%.4f, %.4f]%s: hidden #1 mu = %.4f, A = %.3f,"
                  " resid %.4f -> %.4f (%.1fx)%s"
                  % (ctr, r["lo"], r["hi"], wrap_txt, r["mu"], r["amp"], r["resid0"], r["resid1"], r["ratio1"], extra))
        else:
            print("  anchor %.4f rad, window [%.4f, %.4f]: best hidden mu = %.4f, A = %.3f,"
                  " resid %.4f -> %.4f (%.1fx)  (below %.1fx threshold -> rejected)"
                  % (ctr, r["lo"], r["hi"], r["mu"], r["amp"], r["resid0"], r["resid1"],
                     r["ratio1"] if r["ratio1"] is not None else 1.0, MIN_IMPROV_RATIO))

    found = [r for r in peak_scan if r["accepted"]]
    if found:
        # poor-match warning: baseline-only fit should explain most of the curve variance
        for r in peak_scan:
            wm = (th >= r["lo"]) & (th <= r["hi"])
            denom = float(np.sum((F[wm] - F[wm].mean()) ** 2))
            r["r2base"] = 1.0 - r["resid0"] / denom if denom > 1e-12 else 0.0
        poor = [r for r in peak_scan if r["r2base"] < 0.5]
        if poor:
            print("WARNING: pair matches poorly - the baseline fit explains < 50%% of the curve")
            print("  variance in %d window(s) (R2 = %.2f .. %.2f); accepted hidden candidates may"
                  % (len(poor), min(r["r2base"] for r in poor), max(r["r2base"] for r in poor)))
            print("  be fit artifacts of a low-quality match - treat them as hypotheses only.")
        # headline priority: (1) candidate in the ESTIMATED rotation's window (meta row) -
        # most likely place for the truth; (2) novel shoulder-type candidates; (3) max marginal
        shoulders = [r for r in found if r["shoulder"]]
        reason = ""
        estwin = []
        if est_rad is not None:
            estwin = [r for r in found if circ_dist(r["center"], est_rad % np.pi) < WIN_HALF_RAD]
        if estwin:
            hidden_result = max(estwin, key=lambda r: r["amp"])
            reason = "window of the estimated rotation, largest amplitude"
        elif shoulders:
            hidden_result = max(shoulders, key=lambda r: r["amp"])
            reason = "largest amplitude among SHOULDER/PLATEAU candidates"
        else:
            hidden_result = max(found, key=lambda r: r["ratio"])
            reason = "no shoulder-type candidate -> largest marginal improvement"
        h = hidden_result
        mus_known = h["knowns"]
        amps_known = [float(a) for a in h["coef"][:len(mus_known)]]
        scan_rad, scan_resid = list(h["cand"]), list(h["resids"])
        lm = [th[i] for i in range(1, len(th) - 1) if F[i] > F[i - 1] and F[i] > F[i + 1]]
        kernel_peaks = []
        for _j, _m in enumerate(h["mus_all"]):
            _h = (_j >= len(mus_known))
            _s = not any(circ_dist(_m, _L) < 0.026 for _L in lm)
            kernel_peaks.append(dict(mu=float(_m), amp=float(h["coef"][_j]),
                                     hidden=bool(_h), shoulder=bool(_s),
                                     anchor=float(h["center"])))
        print("\nALL accepted hidden candidates:")
        for r in sorted(found, key=lambda rr: -rr["ratio1"]):
            line = "  anchor %.4f: mu = %.4f rad, A = %.3f, %s, marginal %.1fx (baseline R2 %.2f)" % (
                r["center"], r["mu"], r["amp"],
                "SHOULDER/PLATEAU" if r["shoulder"] else "local max", r["ratio1"], r["r2base"])
            if r["n_hidden"] > 1:
                line += "   [2nd: %.4f rad, A=%.3f, marginal %.1fx]" % (r["mu2"], r["amp2"], r["ratio2"])
            print(line)
        print("\nRESULT (headline: %s): plateau at %.4f rad, A = %.3f, residual %.4f -> %.4f (%.1fx)"
              % (reason, h["mu"], h["amp"], h["resid0"], h["resid1"], h["ratio1"]))
        if h["n_hidden"] > 1:
            print("  second hidden at %.4f rad, A = %.3f (marginal %.1fx)" % (h["mu2"], h["amp2"], h["ratio2"]))
        print("  window [%.4f, %.4f] rad around anchor %.4f rad"
              % (h["lo"], h["hi"], h["center"]))
        print("  -> %s" % ("SHOULDER/PLATEAU (invisible to maxima-based peak detection)"
                           if h["shoulder"] else "local maximum (classical detection would find it)"))
    else:
        hidden_result = None
        print("\nNo hidden plateau accepted: no window improved the residual by >= %.1fx."
              % MIN_IMPROV_RATIO)
else:
    print("Kernel fit skipped (K not available, see previous cell).")


Persistence peaks on [0, pi): [0.0966, 0.4433, 1.129, 1.669]
Scan mode: GT-FREE per-peak scan over 4 window(s)
  anchor 0.0966 rad, window [0.0000, 0.4466]: best hidden mu = 0.2600, A = 0.147, resid 0.8834 -> 0.6538 (1.4x)  (below 2.0x threshold -> rejected)
  anchor 0.4433 rad, window [0.0933, 0.7933]: hidden #1 mu = 0.3433, A = 0.437, resid 3.2651 -> 1.0629 (3.1x)
  anchor 1.1290 rad, window [0.7790, 1.4790]: best hidden mu = 0.9390, A = 0.245, resid 1.6420 -> 1.2027 (1.4x)  (below 2.0x threshold -> rejected)
  anchor 1.6690 rad, window [1.3190, 2.0190]: hidden #1 mu = 1.9606, A = 0.520, resid 5.7772 -> 0.7243 (8.0x)
  variance in 1 window(s) (R2 = 0.47 .. 0.47); accepted hidden candidates may
  be fit artifacts of a low-quality match - treat them as hypotheses only.

ALL accepted hidden candidates:
  anchor 1.6690: mu = 1.9606 rad, A = 0.520, local max, marginal 8.0x (baseline R2 0.47)
  anchor 0.4433: mu = 0.3433 rad, A = 0.437, local max, marginal 3.1x (baseline R2 0.89)

RESULT (

In [180]:
"""Figure 2: folded curve, scan window, kernel model, components, residual (rad)."""
import plotly.graph_objects as go

if K is not None and hidden_result is not None:
    thd = th                                        # folded grid in rad
    mus_all = hidden_result["mus_all"]
    coef_all = hidden_result["coef"]
    model_all = design_matrix(mus_all, th) @ coef_all

    fig2 = go.Figure()
    fig2.add_vrect(x0=hidden_result["lo"], x1=hidden_result["hi"],
                   fillcolor="lightgray", opacity=0.25, line_width=0,
                   annotation_text="scan window (anchor %.4f rad)" % hidden_result["center"],
                   annotation_position="top left")
    fig2.add_trace(go.Scatter(x=thd, y=F, name="folded observed", line=dict(color="black", width=2)))
    fig2.add_trace(go.Scatter(x=thd, y=model_all, name="kernel model", line=dict(color="steelblue", width=1.8)))
    for j, mu in enumerate(mus_all):
        is_hidden = j >= len(mus_known)
        col = "darkred" if is_hidden else "green"
        comp = coef_all[j] * kval(th - mu)
        nm = ("HIDDEN plateau @ %.4f rad (A=%.2f)" % (mu % np.pi, coef_all[j]) if is_hidden
              else "comp %d @ %.4f rad (A=%.2f)" % (j + 1, mu % np.pi, coef_all[j]))
        fig2.add_trace(go.Scatter(x=thd, y=comp, line=dict(dash="dash", width=1.4 if is_hidden else 1.0, color=col),
                                  name=nm))
        fig2.add_vline(x=mu % np.pi, line=dict(color=col, dash="dot", width=1.4 if is_hidden else 1))
        fig2.add_trace(go.Scatter(x=[mu % np.pi], y=[float(np.interp(mu % np.pi, thd, F))],
                                  mode="markers",
                                  marker=dict(symbol="diamond", size=11 if is_hidden else 9, color=col,
                                              line=dict(width=1, color="black")),
                                  hovertemplate="kernel peak @ %{x:.4f} rad<extra></extra>",
                                  showlegend=False))
    res = F - model_all
    fig2.add_trace(go.Scatter(x=thd, y=res, name="residual", yaxis="y2",
                              line=dict(color="red", width=1.2, dash="dot"),
                              hovertemplate="%{x:.4f} rad<br>resid %{y:.5f}<extra></extra>"))
    if peaks is not None:
        prad = np.mod(peaks[:, 0], np.pi)
        fig2.add_trace(go.Scatter(x=prad, y=np.interp(prad, thd, F), mode="markers",
                                  name="persistence peaks (mod pi)",
                                  marker=dict(symbol="x", size=10, color="red", line=dict(width=2)),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    if gt_rad is not None:
        fig2.add_vline(x=gt_rad % np.pi, line=dict(color="green", dash="dash"),
                       annotation_text="GT(est+err) %.4f" % (gt_rad % np.pi), annotation_position="top right")
    if gt_minus_rad is not None:
        fig2.add_vline(x=gt_minus_rad % np.pi, line=dict(color="orange", dash="dash"),
                       annotation_text="GT(est-err) %.4f" % (gt_minus_rad % np.pi), annotation_position="top left")
    for r in found:
        if r is hidden_result or r["n_hidden"] < 1:
            continue
        fig2.add_vline(x=r["mu"] % np.pi, line=dict(color="orange", dash="dot", width=1),
                       annotation_text="cand %.3f" % (r["mu"] % np.pi), annotation_position="top")
        fig2.add_trace(go.Scatter(x=[r["mu"] % np.pi], y=[float(np.interp(r["mu"] % np.pi, thd, F))],
                                  mode="markers",
                                  marker=dict(symbol="diamond", size=9, color="orange",
                                              line=dict(width=1, color="black")),
                                  name="cand %.4f rad (A=%.2f, anchor %.4f)" % (r["mu"] % np.pi, r["amp"], r["center"]),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    fig2.update_layout(title="Folded correlation + kernel fit (windowed)",
                       height=560,
                       legend=dict(orientation="h", y=1.12, font=dict(size=10)),
                       xaxis=dict(title="rotation angle (rad)", range=[0, np.pi],
                                  tickvals=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi],
                                  ticktext=["0", "pi/4", "pi/2", "3pi/4", "pi"]),
                       yaxis=dict(title="normalized correlation"),
                       yaxis2=dict(title="residual", overlaying="y", side="right", showgrid=False),
                       margin=dict(t=80))
    fig2.show()
else:
    print("Figure 2 skipped (no fit results).")


In [181]:
"""Figure 1 update: kernel-fit peaks (green diamonds = persistence-derived, darkred = HIDDEN plateau)."""
if K is not None and hidden_result is not None and curve is not None:
    for p in kernel_peaks:
        col = "darkred" if p["hidden"] else "green"
        xm = [p["mu"] % (2 * np.pi), (p["mu"] + np.pi) % (2 * np.pi)]
        ym = [float(np.interp(t, x, c)) for t in xm]
        fig1.add_trace(go.Scatter(x=xm, y=ym, mode="markers+text",
                                  name=("kernel HIDDEN @ %.3f rad" % (p["mu"] % np.pi)
                                        if p["hidden"] else "kernel peak @ %.3f rad" % (p["mu"] % np.pi)),
                                  marker=dict(symbol="diamond", size=11 if p["hidden"] else 9, color=col,
                                              line=dict(width=1, color="black")),
                                  text=["%.3f (A=%.2f)" % (p["mu"] % np.pi, p["amp"])] * 2,
                                  textposition="bottom center", textfont=dict(size=9, color=col),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    for r in found:
        if r is hidden_result or r["n_hidden"] < 1:
            continue
        xm = [r["mu"] % (2 * np.pi), (r["mu"] + np.pi) % (2 * np.pi)]
        ym = [float(np.interp(t, x, c)) for t in xm]
        fig1.add_trace(go.Scatter(x=xm, y=ym, mode="markers+text",
                                  name="candidate @ %.3f rad (anchor %.3f)" % (r["mu"] % np.pi, r["center"]),
                                  marker=dict(symbol="diamond", size=9, color="orange",
                                              line=dict(width=1, color="black")),
                                  text=["%.3f (A=%.2f)" % (r["mu"] % np.pi, r["amp"])] * 2,
                                  textposition="bottom center", textfont=dict(size=9, color="orange"),
                                  hovertemplate="%{x:.4f} rad<extra></extra>"))
    _redraw()
    print("Figure 1: green = persistence-derived peaks, darkred = headline plateau, orange = other accepted candidates.")
else:
    print("Figure 1 update skipped (no fit results).")


Figure 1: green = persistence-derived peaks, darkred = headline plateau, orange = other accepted candidates.


In [182]:
"""Combined summary (rad): deduplicated persistence peaks + per-peak kernel scan, GT-free."""
if K is not None:
    print("=== Persistence peaks on [0, pi) ===   (%d unique)" % len(known_all))
    for p in known_all:
        row = peaks[np.argmin(circ_dist(peaks[:, 0] % np.pi, p))]
        print("  %9.4f rad  corr=%.3f  levelPot=%.4f" % (p, row[1], row[3]))
    if hidden_result is not None:
        print("\n=== Per-peak kernel scan (window = anchor +/- %.2f rad, threshold %.1fx, max %d hidden, GT-free) ==="
              % (WIN_HALF_RAD, MIN_IMPROV_RATIO, MAX_HIDDEN))
        print("  %9s %5s %12s %7s %-24s %s %s" % ("anchor", "hid#", "mu (rad)", "A", "resid step", "marginal", "type"))
        for r in peak_scan:
            if not r["accepted"]:
                print("  %9.4f  [rejected] best mu = %.4f, A = %.3f, resid %.4f -> %.4f (%.1fx)  < %.1fx"
                      % (r["center"], r["mu"], r["amp"], r["resid0"], r["resid1"],
                         r["ratio1"] if r["ratio1"] is not None else 1.0, MIN_IMPROV_RATIO))
                continue
            print("  %9.4f %5d %12.4f %7.3f %-24s %.1fx   %s"
                  % (r["center"], 1, r["mu"] % np.pi, r["amp"],
                     "%.4f -> %.4f" % (r["resid0"], r["resid1"]), r["ratio1"],
                     "SHOULDER" if r["shoulder"] else "local max"))
            if r["n_hidden"] > 1:
                print("  %9.4f %5d %12.4f %7.3f %-24s %.1fx   %s"
                      % (r["center"], 2, r["mu2"] % np.pi, r["amp2"],
                         "%.4f -> %.4f" % (r["resid1"], r["resid2"]), r["ratio2"],
                         "SHOULDER" if r["shoulder2"] else "local max"))
        h = hidden_result
        print("\n=== Kernel-fit peaks (best window [%.4f, %.4f] rad around anchor %.4f, %d hidden) ==="
              % (h["lo"], h["hi"], h["center"], h["n_hidden"]))
        print("  %9s %8s %-16s %s" % ("mu (rad)", "A", "source", "type"))
        _hn = 0
        for p in kernel_peaks:
            if p["hidden"]:
                _hn += 1
                src = "HIDDEN %d" % _hn if h["n_hidden"] > 1 else "HIDDEN plateau"
            else:
                src = "persistence"
            print("  %9.4f %8.3f %-16s %s" % (p["mu"] % np.pi, p["amp"], src,
                  "shoulder" if p["shoulder"] else "local max"))
        if h["n_hidden"] > 1:
            print("  residual %.4f -> %.4f -> %.4f  (marginal %.1fx + %.1fx)"
                  % (h["resid0"], h["resid1"], h["resid2"], h["ratio1"], h["ratio2"]))
        else:
            print("  residual %.4f -> %.4f (%.1fx improvement)" % (h["resid0"], h["resid1"], h["ratio1"]))
        if gt_rad is not None and est_rad is not None:
            g1 = float(gt_rad % np.pi)                          # convention: err = true - est
            g2 = float(np.mod(2.0 * est_rad - gt_rad, np.pi))   # convention: err = est - true
            d1, d2 = circ_dist(g1, h["mu"]), circ_dist(g2, h["mu"])
            d = float(min(d1, d2))
            conv = "err = est - true" if d2 < d1 else "err = true - est"
            print("  GT validation (debug; error sign convention depends on the writing pipeline):")
            print("    GT(est+err) = %.4f rad, GT(est-err) = %.4f rad" % (g1, g2))
            print("    hidden plateau %.4f rad is %.4f rad (~%.2f deg) from the nearer GT (%s)"
                  % (h["mu"], d, np.rad2deg(d), conv))
            print("    -> %s" % ("CORRECT pick (within 0.035 rad)" if d < 0.035
                                 else "NOT near either GT - treat candidates as hypotheses only"))
            print("    all accepted candidates vs the nearer GT:")
            for r in sorted(found, key=lambda rr: -rr["ratio1"]):
                dd = float(min(circ_dist(g1, r["mu"]), circ_dist(g2, r["mu"])))
                print("      anchor %.4f: mu = %.4f rad -> %.4f rad (~%.2f deg) away %s"
                      % (r["center"], r["mu"], dd, np.rad2deg(dd),
                         "[near GT]" if dd < 0.035 else ""))
        print("\nDifference: the kernel fit confirms the persistence peaks inside the best window")
        print("and adds %d hidden plateau(s) that classical detection cannot see (shoulder); each"
              % h["n_hidden"])
        print("level had to justify itself by >= %.1fx marginal residual improvement. The scan never"
              % MIN_IMPROV_RATIO)
        print("uses GT - it probes every persistence peak. See the residual trace in Figure 2.")
    else:
        print("\nNo hidden plateau accepted: every window stayed below the %.1fx residual threshold."
              % MIN_IMPROV_RATIO)
else:
    print("Summary skipped (no fit results).")


=== Persistence peaks on [0, pi) ===   (4 unique)
     0.0966 rad  corr=0.756  levelPot=0.5723
     0.4433 rad  corr=1.000  levelPot=1.0000
     1.1290 rad  corr=0.315  levelPot=0.0992
     1.6690 rad  corr=0.834  levelPot=0.6949

=== Per-peak kernel scan (window = anchor +/- 0.35 rad, threshold 2.0x, max 2 hidden, GT-free) ===
     anchor  hid#     mu (rad)       A resid step               marginal type
     0.0966  [rejected] best mu = 0.2600, A = 0.147, resid 0.8834 -> 0.6538 (1.4x)  < 2.0x
     0.4433     1       0.3433   0.437 3.2651 -> 1.0629         3.1x   local max
     1.1290  [rejected] best mu = 0.9390, A = 0.245, resid 1.6420 -> 1.2027 (1.4x)  < 2.0x
     1.6690     1       1.9606   0.520 5.7772 -> 0.7243         8.0x   local max

=== Kernel-fit peaks (best window [0.0933, 0.7933] rad around anchor 0.4433, 1 hidden) ===
   mu (rad)        A source           type
     0.0966    0.686 persistence      local max
     0.4433    0.891 persistence      local max
     0.3433    0.

## Notes & pitfalls

- **pi-periodicity:** the resampled Fourier magnitude is even in azimuth, so $C(\theta)$ is (bit-)exactly periodic in $\pi$. The folded trace is lossless; angles are physical rotations modulo $\pi$ rad.
- **GT derivation:** `registration_meta.csv` stores the *estimated* rotation and the GT *error* (in degrees); GT = estimated + error, displayed in radians. Verify the pair is the one you ran.
- **GT-free by design:** the scan probes one window per persistence peak; GT only scores the pick (meta row). If the data dir mixes runs (compare file mtimes), the meta row may belong to another pair - the plateau is still found, only the validation line is then meaningless.
- **Gaussian fits will mislead:** the true peak kernel is NOT Gaussian (band-limited correlation: sharp core + tails + side lobes, e.g. a strong lobe near $\pi/2$ rad in one earlier pair). A Gaussian mixture invents spurious components there. Use the kernel from the coefficients.
- **Shoulders are invisible to peak detectors** (local-max / persistence alike): they are not local maxima. Only model fitting (kernel subtraction) reveals them.
- **Resolution limit:** two rotations closer than ~ the kernel core width (roughly 0.05-0.15 rad) cannot be separated; the scan enforces `MIN_SEP_RAD` separation from known components.
- **The kernel is per scan pair:** it is the autocorrelation of the *reference* (scan 2) descriptor. Recompute it for each debug run (the notebook does this automatically).
- Missing files? Run fs2d with debug + `useDirect=true` to get the coefficient files; without them the notebook falls back to an empirical kernel (dominant peak shape).
